# U-Net: phân đoạn ảnh BUSI / Image segmentation

**Người thực hành / Learner:** Trương Minh Hoàng · **MSSV / Student ID:** 25410008

Bài thực hành để đọc, chỉnh sửa và giải thích quy trình học sâu. / A learning exercise in understanding and improving a deep-learning pipeline.

**Nguồn / Provenance:** Bản Colab tham khảo có tên gốc `LAB6_21540008.ipynb`. Thông tin người thực hành ở trên được chuẩn hóa cho portfolio; tên nguồn được giữ để phân biệt việc thực hành với tác giả ban đầu.

**Input → Output:** ảnh siêu âm và mask → xác suất phân đoạn theo pixel. Cần dataset BUSI hợp lệ tại đường dẫn cấu hình; chưa chạy lại huấn luyện trên dữ liệu BUSI trong đợt này.

**Tôi luyện tập:** ghép cặp ảnh/mask, gộp nhãn, encoder–decoder, skip connection, Dice loss và IoU. / Practice image-mask pairing, segmentation networks and overlap metrics.

# Giải thích quy trình bài Lab: Phân đoạn hình ảnh siêu âm vú (U-Net)

Bài Lab này thực hiện quy trình học sâu từ đầu đến cuối:
1. **Chuẩn bị dữ liệu**: Kết nối Google Drive, giải nén và tiền xử lý ảnh (gộp các mặt nạ - mask).
2. **Xây dựng Pipeline**: Resize ảnh về $256 \times 256$, chuẩn hóa dữ liệu về dạng mảng NumPy và chia tập Train/Test.
3. **Kiến trúc mô hình**: Xây dựng mạng U-Net (Encoder-Decoder) với các khối Convolution và Skip Connections.
4. **Huấn luyện**: Sử dụng hàm loss Dice và chỉ số BinaryIoU để tối ưu mô hình.
5. **Đánh giá**: Vẽ biểu đồ lịch sử huấn luyện và dự đoán thử nghiệm trên tập Test.

In [ ]:
# @title
from google.colab import drive
drive.mount("/content/drive/")

### Bước 1: Kết nối dữ liệu
Ô mã phía trên dùng để kết nối Google Colab với Google Drive để truy cập tệp tin dataset.

In [ ]:
import shutil
!unzip "/content/drive/MyDrive/Dataset_BUSI.zip" -d "/content/"

### Bước 2: Giải nén Dataset
Sử dụng lệnh shell `unzip` để giải nén bộ dữ liệu Dataset_BUSI vào môi trường chạy tạm thời của Colab.

In [ ]:
import os, re, cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from tensorflow import keras
import tensorflow as tf
SEED = 42
tf.keras.utils.set_random_seed(SEED)
root_path = Path(os.environ.get('BUSI_DATA_DIR', '/content/Dataset_BUSI_with_GT'))
if not root_path.is_dir():
    raise FileNotFoundError('Set BUSI_DATA_DIR to an extracted BUSI dataset directory.')
# Match masks by the exact image stem; preserve every original data file.
pairs = []
for image_path in sorted(root_path.rglob('*.png')):
    if '_mask' in image_path.stem:
        continue
    masks = sorted(p for p in image_path.parent.glob(image_path.stem + '_mask*.png')
                   if re.fullmatch(re.escape(image_path.stem) + r'_mask(?:_\d+)?', p.stem))
    if not masks:
        raise ValueError(f'Missing mask for {image_path.name}')
    pairs.append((image_path, masks))
assert pairs, 'No paired images found'
print('Paired images:', len(pairs))

### Bước 3: Tiền xử lý Mask (Gộp nhãn)
Một số ảnh có nhiều tệp mask riêng lẻ (ví dụ: mask_1, mask_2). Đoạn code này sẽ gộp tất cả các mask của cùng một ảnh thành một mặt nạ duy nhất bằng phép toán `bitwise_or`.

In [ ]:
IMG_WIDTH = IMG_HEIGHT = 256
X_data, Y_data = [], []
for image_path, masks in pairs:
    image = cv2.imread(str(image_path), cv2.IMREAD_GRAYSCALE)
    if image is None:
        raise ValueError(f'Unreadable image: {image_path.name}')
    combined = np.zeros_like(image, dtype=np.uint8)
    for mask_path in masks:
        mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
        if mask is None or mask.shape != image.shape:
            raise ValueError(f'Invalid mask: {mask_path.name}')
        combined = cv2.bitwise_or(combined, mask)
    X_data.append(cv2.resize(image, (IMG_WIDTH, IMG_HEIGHT)))
    Y_data.append(cv2.resize(combined, (IMG_WIDTH, IMG_HEIGHT), interpolation=cv2.INTER_NEAREST))
assert len(X_data) == len(Y_data)

### Bước 4: Đọc và Resize dữ liệu
Đọc toàn bộ ảnh siêu âm và ảnh mask, sau đó thay đổi kích thước về chuẩn $256 \times 256$ để đưa vào mô hình.

In [ ]:
import random
def visual(number_img= 3):
    random_indices = random.sample(range(len(X_data)), number_img)
    for i, idx in enumerate(random_indices, 1):
        plt.figure(figsize=(10, 5))
        plt.subplot(131)
        plt.imshow(X_data[idx], cmap='gray')
        # plt.title(f'X_train - Thứ tự: {idx}')
        plt.title(f'Image Siêu Âm')
        plt.subplot(132)
        plt.imshow(Y_data[idx], cmap='gray')
        # plt.title(f'Y_train - Thứ tự: {idx}')
        plt.title(f'Image Mask')
        plt.subplot(133)
        plt.imshow(X_data[idx], cmap='gray')
        plt.imshow(Y_data[idx], alpha=0.5)
        plt.title(f"Lesion Label ")
        print(f"name {idx}")
        # plt.savefig(f'{idx}.pdf', format='pdf')
        plt.show()
visual(5)

### Bước 5: Kiểm tra dữ liệu trực quan
Hiển thị ngẫu nhiên một vài cặp ảnh (Ảnh gốc - Ảnh Mask - Ảnh đè nhãn) để đảm bảo dữ liệu đã được load đúng.

In [ ]:
X_data = np.repeat(np.asarray(X_data, dtype=np.float32)[..., None], 3, axis=-1) / 255.0
Y_data = (np.asarray(Y_data) > 0).astype(np.float32)[..., None]
assert X_data.shape[:3] == Y_data.shape[:3]
assert 0 <= X_data.min() <= X_data.max() <= 1
print('Images:', X_data.shape, '| Masks:', Y_data.shape)

### Bước 6: Chuyển đổi Tensor
Chuyển danh sách ảnh thành mảng NumPy 4 chiều, chuẩn hóa kiểu dữ liệu `float32` để phù hợp với yêu cầu của TensorFlow/Keras.

In [ ]:
from sklearn.model_selection import train_test_split
X_temp, X_test, Y_temp, Y_test = train_test_split(X_data, Y_data, test_size=0.2, random_state=SEED)
X_train, X_valid, Y_train, Y_valid = train_test_split(X_temp, Y_temp, test_size=0.25, random_state=SEED)
print('Train/validation/test:', len(X_train), len(X_valid), len(X_test))
# Image-level split only: a patient-level split requires reliable patient identifiers.

### Bước 7: Chia tập dữ liệu
Chia theo ảnh: 60% train, 20% validation, 20% test. Chia theo bệnh nhân cần mã định danh bệnh nhân đáng tin cậy.

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Conv2D, MaxPooling2D, Dropout, concatenate, Conv2DTranspose, BatchNormalization, Activation
from tensorflow.keras.models import Model

def Conv2dBlock(inputTensor, numFilters, kernelSize=3, doBatchNorm=True):
    """
    Hàm tạo khối Convolution kép (Double Convolution Block).
    Tham số:
    - inputTensor: Tensor đầu vào của khối.
    - numFilters: Số lượng bộ lọc (kernels) cho lớp Convolution.
    - kernelSize: Kích thước cửa sổ trích xuất đặc trưng (mặc định 3x3).
    - doBatchNorm: Boolean, nếu True sẽ áp dụng chuẩn hóa lớp (Batch Normalization).
    """
    # Lớp Conv thứ nhất
    x = Conv2D(filters=numFilters, kernel_size=(kernelSize, kernelSize),
               kernel_initializer='he_normal', padding='same')(inputTensor)
    if doBatchNorm:
        x = BatchNormalization()(x)
    x = Activation('relu')(x)

    # Lớp Conv thứ hai
    x = Conv2D(filters=numFilters, kernel_size=(kernelSize, kernelSize),
               kernel_initializer='he_normal', padding='same')(x)
    if doBatchNorm:
        x = BatchNormalization()(x)
    x = Activation('relu')(x)

    return x

def Unet_base(inputImage, numFilters=16, dropout=0.1, doBatchNorm=True):
    """
    Xây dựng toàn bộ cấu trúc mạng U-Net.
    Tham số:
    - inputImage: Tensor ảnh đầu vào (thường là 256x256x3).
    - numFilters: Số lượng bộ lọc cơ sở (sẽ nhân đôi sau mỗi tầng Encoder).
    - dropout: Tỷ lệ loại bỏ nơ-ron để chống quá khớp (Overfitting).
    - doBatchNorm: Áp dụng chuẩn hóa để tăng tốc độ hội tụ.
    """

    # 1. ENCODER PATH (Trích xuất đặc trưng và giảm kích thước)
    # Tầng 1: 256x256 -> 128x128
    c1 = Conv2dBlock(inputImage, numFilters * 1, kernelSize=3, doBatchNorm=doBatchNorm)
    p1 = MaxPooling2D((2, 2))(c1)
    p1 = Dropout(dropout)(p1)

    # Tầng 2: 128x128 -> 64x64
    c2 = Conv2dBlock(p1, numFilters * 2, kernelSize=3, doBatchNorm=doBatchNorm)
    p2 = MaxPooling2D((2, 2))(c2)
    p2 = Dropout(dropout)(p2)

    # Tầng 3: 64x64 -> 32x32
    c3 = Conv2dBlock(p2, numFilters * 4, kernelSize=3, doBatchNorm=doBatchNorm)
    p3 = MaxPooling2D((2, 2))(c3)
    p3 = Dropout(dropout)(p3)

    # Tầng 4: 32x32 -> 16x16
    c4 = Conv2dBlock(p3, numFilters * 8, kernelSize=3, doBatchNorm=doBatchNorm)
    p4 = MaxPooling2D((2, 2))(c4)
    p4 = Dropout(dropout)(p4)

    # 2. BOTTLENECK (Phần đáy của chữ U - Nén thông tin ở mức sâu nhất)
    c5 = Conv2dBlock(p4, numFilters * 16, kernelSize=3, doBatchNorm=doBatchNorm)

    # 3. DECODER PATH (Khôi phục kích thước và kết hợp Skip Connections)
    # Tầng 6: 16x16 -> 32x32
    u6 = Conv2DTranspose(numFilters * 8, (3, 3), strides=(2, 2), padding='same')(c5)
    u6 = concatenate([u6, c4]) # Skip Connection: nối thông tin từ Encoder
    u6 = Dropout(dropout)(u6)
    c6 = Conv2dBlock(u6, numFilters * 8, kernelSize=3, doBatchNorm=doBatchNorm)

    # Tầng 7: 32x32 -> 64x64
    u7 = Conv2DTranspose(numFilters * 4, (3, 3), strides=(2, 2), padding='same')(c6)
    u7 = concatenate([u7, c3])
    u7 = Dropout(dropout)(u7)
    c7 = Conv2dBlock(u7, numFilters * 4, kernelSize=3, doBatchNorm=doBatchNorm)

    # Tầng 8: 64x64 -> 128x128
    u8 = Conv2DTranspose(numFilters * 2, (3, 3), strides=(2, 2), padding='same')(c7)
    u8 = concatenate([u8, c2])
    u8 = Dropout(dropout)(u8)
    c8 = Conv2dBlock(u8, numFilters * 2, kernelSize=3, doBatchNorm=doBatchNorm)

    # Tầng 9: 128x128 -> 256x256
    u9 = Conv2DTranspose(numFilters * 1, (3, 3), strides=(2, 2), padding='same')(c8)
    u9 = concatenate([u9, c1])
    u9 = Dropout(dropout)(u9)
    c9 = Conv2dBlock(u9, numFilters * 1, kernelSize=3, doBatchNorm=doBatchNorm)

    # 4. OUTPUT LAYER (Phân loại từng Pixel)
    # Sử dụng sigmoid để đưa ra xác suất (0 hoặc 1) cho việc pixel đó thuộc về khối u hay không
    output = tf.keras.layers.Conv2D(1, (1, 1), activation='sigmoid')(c9)

    return Model(inputs=[inputImage], outputs=[output])

### Bước 8: Định nghĩa kiến trúc U-Net
Xây dựng các hàm bổ trợ:
- `Conv2dBlock`: Khối tích chập kép có Batch Normalization.
- `Unet_base`: Cấu trúc Encoder (giảm chiều) và Decoder (tăng chiều) đặc trưng của U-Net.

In [ ]:
Unet = Unet_base(Input((256, 256, 3)), dropout= 0.1)

In [ ]:
Unet.summary()

In [ ]:
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
callbacks = [ReduceLROnPlateau(monitor='val_BinaryIoU', mode='max', patience=10, factor=0.5),
             EarlyStopping(monitor='val_BinaryIoU', mode='max', patience=20, restore_best_weights=True)]
BinaryIoU = tf.keras.metrics.BinaryIoU(name='BinaryIoU', threshold=0.5)
def dice_loss(y_true, y_pred):
    y_true = tf.cast(y_true, tf.float32)
    y_pred = tf.cast(y_pred, tf.float32)
    intersection = tf.reduce_sum(y_true * y_pred, axis=(1, 2, 3))
    denominator = tf.reduce_sum(y_true + y_pred, axis=(1, 2, 3))
    return tf.reduce_mean(1 - (2 * intersection + 1e-6) / (denominator + 1e-6))

In [ ]:
Unet.compile(optimizer='adam', loss=dice_loss, metrics=[BinaryIoU])
History = Unet.fit(X_train, Y_train, validation_data=(X_valid, Y_valid),
                   batch_size=8, epochs=100, callbacks=callbacks)
Unet.save('busi_unet_25410008.keras')

### Bước 9: Huấn luyện mô hình
Thiết lập hàm Loss (Dice), Optimizer (Adam) và bắt đầu quá trình Training với 100 Epochs.

In [ ]:
print("Test data")
results = Unet.evaluate(X_test, Y_test, batch_size=32)
print("Test Dice loss and BinaryIoU:", results)

In [ ]:
def visual(history, model_name):
    plt.figure(figsize=(10, 8))
    plt.subplot(2, 1, 1)
    plt.plot(history.history['loss'],'r', label='Training Loss')
    plt.plot(history.history['val_loss'],'r--', label='Validation Loss')
    plt.title(f'Loss - {model_name}')
    plt.legend()
    plt.subplot(2, 1, 2)
    plt.plot(history.history['BinaryIoU'],'b', label='Training acc')
    plt.plot(history.history['val_BinaryIoU'],'b--', label='Validation acc')
    plt.title(f'Binary IoU - {model_name}')
    plt.legend()
    plt.tight_layout()
    plt.show()

visual(History, "Unet")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import random

y_pred_Unet = Unet.predict(X_test)
y_pred_Unet = (y_pred_Unet > 0.5).astype(np.float32)

num_samples = 5
random_indices = random.sample(range(len(X_test)), num_samples)
fig, axs = plt.subplots(nrows=num_samples, ncols=3, figsize=(10, num_samples * 2))

for i, index in enumerate(random_indices):
    test_image = X_test[index, :, :, 0]
    data_image = Y_test[index, :, :, 0]
    pred_mask_Unet = y_pred_Unet[index, :, :, 0]

    axs[i, 0].imshow(test_image, cmap='summer')
    axs[i, 0].set_title("Ảnh X_test")
    axs[i, 1].imshow(data_image, cmap='YlGn')
    axs[i, 1].set_title("Nhãn thực tế")
    axs[i, 2].imshow(pred_mask_Unet, cmap='YlGn')
    axs[i, 2].set_title("Unet dự đoán")

for ax in axs.flat:
    ax.axis('on')

plt.tight_layout()
plt.show()

### Bước 10: Dự đoán và Hiển thị kết quả
Sử dụng mô hình đã huấn luyện để dự đoán trên tập dữ liệu chưa từng thấy (`X_test`) và so sánh trực quan với nhãn gốc.

## Những điểm tôi chỉnh sửa / Changes I practiced

Ghép ảnh/mask bằng tên gốc; gộp mask trong bộ nhớ; không xóa dữ liệu nguồn; resize mask bằng nearest-neighbor; chuẩn hóa ảnh về [0,1]; tách validation trước huấn luyện; dùng Dice loss tường minh; đặt callback theo hướng tối đa hóa IoU.

Bản nguồn chia theo ảnh. Khi có mã bệnh nhân, cần chia theo bệnh nhân để kiểm tra khả năng khái quát. Các ghi chú quy trình phía trên thuộc bài gốc; code đã cập nhật dùng train/validation/test 60/20/20. Chưa có metric BUSI mới để báo cáo.

## Kiểm tra kỹ thuật / Technical check

**SMOKE_PASS:** kiến trúc có 2.164.593 tham số; đầu ra tensor `(1, 256, 256, 1)` hữu hạn. Dice loss bằng 0 với cặp mask hoàn hảo (toàn 0 hoặc toàn 1), và trên 0,99 khi dự đoán toàn 0 cho mask toàn 1. Kiểm tra dùng tensor tạo mẫu, chưa đo hiệu năng trên BUSI.